# 🦺 PPE Safety Detection Demo
# This notebook demonstrates how to detect Personal Protective Equipment (PPE)
# using a pretrained Faster R-CNN model from Torchvision.


In [1]:
import cv2
import torch
import torchvision.transforms as T
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

# Load pretrained weights
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights)
model.eval()

transform = T.Compose([T.ToTensor()])


In [2]:
def detect_ppe(image_path):
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    tensor = transform(img_rgb)
    with torch.no_grad():
        predictions = model([tensor])
    return predictions, img


In [3]:
# Run detection on a sample image
preds, img = detect_ppe("../data/raw/sample.jpg")

for box, score, label in zip(preds[0]['boxes'], preds[0]['scores'], preds[0]['labels']):
    if score > 0.7:  # confidence threshold
        x1, y1, x2, y2 = box.int().tolist()
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, f"Label {label}", (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

cv2.imshow("PPE Detection", img)
cv2.waitKey(0)
cv2.destroyAllWindows()
